In [ ]:
from pathlib import Path
import json
import itertools

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from autogluon.tabular import TabularPredictor
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    precision_recall_curve,
)

LABEL = "log2FC"
DATA_ROOT = Path("/media/volume/sirna-features")
OUT_DIR = DATA_ROOT / "training" / "autogluon_stack_level_runs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

DATASETS = {
    "0": DATA_ROOT / "dataset0" / "raw_data_0.csv",
    "1": DATA_ROOT / "dataset1" / "raw_data_1.csv",
    "2": DATA_ROOT / "dataset2" / "raw_data_2.csv",
    "3": DATA_ROOT / "dataset3" / "raw_data_3.csv",
    "4": DATA_ROOT / "dataset4" / "raw_data_4.csv",
    "6N": DATA_ROOT / "dataset6_7" / "raw_data_6_noxform.csv",
    "6R": DATA_ROOT / "dataset6_7" / "raw_data_6_rescaled.csv",
    "7N": DATA_ROOT / "dataset6_7" / "raw_data_7_noxform.csv",
    "7R": DATA_ROOT / "dataset6_7" / "raw_data_7_rescaled.csv",
}


# First, the evaluation metric should be optimized beforehand (code not shown).
# Next, the time and stack level should be optimized, with the seed varied.
# Autogluon did not require GPUs as training was fast
#

DATASET_IDS = ["0", "1", "2", "3", "4", "6N", "6R", "7N", "7R"]
SEEDS = [2162, 4389, 5836, 7414]
TIME_LIMITS = [295, 355, 370, 780, 980]
STACK_LEVELS = [4, 5, 6]
EVAL_METRIC = "roc_auc_ovo_macro"
NUM_BAG_FOLDS = 2
NUM_BAG_SETS = 1

SEARCH_RUNS = [
    {
        "dataset_id": dataset_id,
        "seed": seed,
        "time_limit": time_limit,
        "stack_level": stack_level,
    }
    for dataset_id, seed, time_limit, stack_level in itertools.product(
        DATASET_IDS, SEEDS, TIME_LIMITS, STACK_LEVELS
    )
]




def run_stack_experiment(dataset_id: str, seed: int, time_limit: int, stack_level: int):
    csv_path = DATASETS[dataset_id]
    df = pd.read_csv(csv_path)

    X = df.drop(columns=[LABEL])
    y = df[LABEL].astype(int).copy()

    run_name = f"{dataset_id}_seed{seed}_stack{stack_level}_t{time_limit}"
    model_dir = OUT_DIR / f"AutogluonModels_Stack_{run_name}"
    model_dir.mkdir(parents=True, exist_ok=True)

    predictor = TabularPredictor(
        label=LABEL,
        problem_type="binary",
        eval_metric=EVAL_METRIC,
        path=str(model_dir),
    ).fit(
        train_data=df,
        presets="best_quality",
        time_limit=time_limit,
        holdout_frac=0.05,
        num_bag_folds=2,
        num_bag_sets=1,
        num_stack_levels=stack_level,
        ag_args_fit={"num_cpus": 64, "seed": seed},
    )

    leaderboard = predictor.leaderboard(df, silent=True)
    leaderboard.to_csv(model_dir / "leaderboard.csv", index=False)

    fit_summary = predictor.fit_summary(verbosity=0)
    with open(model_dir / "fit_summary.json", "w") as f:
        json.dump(fit_summary, f, indent=2, default=str)

    y_pred = predictor.predict(X)
    y_proba = predictor.predict_proba(X, as_pandas=True)

    if 1 in y_proba.columns:
        pos_col = 1
    elif "1" in y_proba.columns:
        pos_col = "1"
    else:
        pos_col = y_proba.columns[-1]

    y_score = y_proba[pos_col].to_numpy()
    y_true = y.to_numpy()

    precision, recall, thresholds = precision_recall_curve(y_true, y_score)
    prc_auc = average_precision_score(y_true, y_score)

    prc_df = pd.DataFrame({"recall": recall, "precision": precision})
    if len(thresholds) == len(prc_df) - 1:
        prc_df.loc[: len(thresholds) - 1, "threshold"] = thresholds
    prc_df.to_csv(model_dir / "prc_curve.csv", index=False)

    cm = confusion_matrix(y_true, y_pred.astype(int))
    report = classification_report(y_true, y_pred.astype(int), digits=4)

    with open(model_dir / "metrics.txt", "w") as f:
        f.write(f"Dataset ID: {dataset_id}\n")
        f.write(f"Seed: {seed}\n")
        f.write(f"Time limit: {time_limit}\n")
        f.write(f"Stack level: {stack_level}\n")
        f.write(f"PRC AUC: {prc_auc:.6f}\n")
        f.write("Confusion Matrix:\n")
        f.write(np.array2string(cm))
        f.write("\n\nClassification Report:\n")
        f.write(report)

    with open(model_dir / "run_summary.json", "w") as f:
        json.dump(
            {
                "dataset_id": dataset_id,
                "csv_path": str(csv_path),
                "seed": seed,
                "time_limit": time_limit,
                "stack_level": stack_level,
                "eval_metric_internal": EVAL_METRIC,
                "prc_auc": float(prc_auc),
                "best_model": predictor.model_best,
            },
            f,
            indent=2,
        )

    plt.figure(figsize=(6, 5))
    plt.plot(recall, precision, lw=2, label=f"{dataset_id} | AUPRC={prc_auc:.3f}")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(f"Precision-Recall Curve | {run_name}")
    plt.legend()
    plt.tight_layout()
    plt.savefig(model_dir / "prc_curve.png", dpi=200)
    plt.show()

    print(f"Finished {run_name}")
    print(f"Best model: {predictor.model_best}")
    print(f"PRC AUC: {prc_auc:.6f}")

    return {
        "dataset_id": dataset_id,
        "seed": seed,
        "time_limit": time_limit,
        "stack_level": stack_level,
        "prc_auc": float(prc_auc),
        "best_model": predictor.model_best,
        "model_dir": str(model_dir),
    }


all_results = []
for run in RUNS:
    result = run_stack_experiment(**run)
    all_results.append(result)

results_df = pd.DataFrame(all_results).sort_values(
    by=["dataset_id", "prc_auc"], ascending=[True, False]
)
results_df.to_csv(OUT_DIR / "stack_level_summary.csv", index=False)
results_df
